# PII Redaction Pipeline for Indian DRHPs

**Approach:** Presidio Analyzer (SpaCy `en_core_web_trf` transformer backbone) + custom regex recognizers + Faker synthetic replacement.

| Cell | Purpose |
|------|---------|
| 1 | System installs & pip dependencies |
| 2 | Python imports |
| 3 | Presidio engine, custom recognizers & allowlist |
| 4 | Faker registry (consistent fake ↔ real mapping) |
| 5 | Core redaction engine (`redact_text_block`, `process_image_ocr`, `process_docx`) |
| 6 | Evaluation function (`score`) & human-verified ground truth |
| 7 | Execute: redact full document + print evaluation report |

## 1. System Setup & Dependencies

In [ ]:
!apt-get update && apt-get install -y tesseract-ocr
!pip install python-docx presidio-analyzer presidio-anonymizer Faker pytesseract spacy spacy-transformers Pillow pandas
!python -m spacy download en_core_web_trf

## 2. Imports

In [ ]:
import json
import logging
import re
import io
from collections import Counter

import docx
import pytesseract
import pandas as pd
from PIL import Image, ImageDraw
from faker import Faker
from presidio_analyzer import AnalyzerEngine, PatternRecognizer, Pattern
from presidio_analyzer.nlp_engine import NlpEngineProvider

logging.getLogger("presidio-analyzer").setLevel(logging.ERROR)

## 3. Presidio Engine, Custom Recognizers & Allowlist

In [ ]:
# ── NLP engine (SpaCy transformer) ──────────────────────────────────────────
provider = NlpEngineProvider(nlp_configuration={
    "nlp_engine_name": "spacy",
    "models": [{"lang_code": "en", "model_name": "en_core_web_trf"}]
})
nlp_engine = provider.create_engine()
analyzer = AnalyzerEngine(nlp_engine=nlp_engine, supported_languages=["en"])

# ── Indian ID recognizers ────────────────────────────────────────────────────
# PAN
analyzer.registry.add_recognizer(PatternRecognizer(
    supported_entity="IN_PAN",
    patterns=[Pattern("pan", r"\b[A-Z]{5}[0-9]{4}[A-Z]\b", 0.95)]))

# Aadhaar (starts with 2-9)
analyzer.registry.add_recognizer(PatternRecognizer(
    supported_entity="IN_AADHAAR",
    patterns=[Pattern("aadhaar", r"\b[2-9]\d{3}\s\d{4}\s\d{4}\b", 0.95)]))

# Date of Birth (with label prefix, Hindi + English)
analyzer.registry.add_recognizer(PatternRecognizer(
    supported_entity="DATE_OF_BIRTH",
    patterns=[Pattern("dob", r"(?i)(?:DOB|Date of Birth|D\.O\.B\.|\u091c\u0928\u094d\u092e \u0924\u093f\u0925\u093f)\s*[:\-]?\s*([0-3]\d[/\-\.][0-1]\d[/\-\.](?:19|20)\d{2})", 0.95)]))

# Indian phone numbers (+91, landlines, mobiles)
analyzer.registry.add_recognizer(PatternRecognizer(
    supported_entity="PHONE_NUMBER",
    patterns=[Pattern("in_phone", r"(?:\+?\s*91[\-\s]?)?(?:\(?0\d{2,4}\)?[\-\s]*)?[6-9]\d{9}|\b0\d{2,4}[\-\s]?\d{6,8}\b", 0.85)]))

# CIN (Corporate Identity Number)
analyzer.registry.add_recognizer(PatternRecognizer(
    supported_entity="IN_CIN",
    patterns=[Pattern("cin", r"\b[LU]\d{5}[A-Z]{2}\d{4}[A-Z]{3}\d{6}\b", 0.95)]))

# SEBI Registration Number
analyzer.registry.add_recognizer(PatternRecognizer(
    supported_entity="IN_SEBI_REG",
    patterns=[Pattern("sebi", r"\bIN[A-Z]{1,3}\d{7,9}\b", 0.95)]))

# PIN Code (6-digit)
analyzer.registry.add_recognizer(PatternRecognizer(
    supported_entity="IN_PIN_CODE",
    patterns=[Pattern("pin", r"\b[1-9][0-9]{5}\b", 0.4)]))

# SSN (assignment requirement — unlikely in Indian docs but required)
analyzer.registry.add_recognizer(PatternRecognizer(
    supported_entity="US_SSN",
    patterns=[Pattern("ssn", r"\b\d{3}-\d{2}-\d{4}\b", 0.95)]))

# Credit card numbers (assignment requirement)
analyzer.registry.add_recognizer(PatternRecognizer(
    supported_entity="CREDIT_CARD",
    patterns=[Pattern("cc", r"\b(?:4\d{3}|5[1-5]\d{2}|3[47]\d{2}|6(?:011|5\d{2}))[\s\-]?\d{4}[\s\-]?\d{4}[\s\-]?\d{4}\b", 0.90)]))

# Slashed contact names (e.g. "Kishan Rastogi / Abhijit Diwan")
analyzer.registry.add_recognizer(PatternRecognizer(
    supported_entity="PERSON",
    patterns=[Pattern("slashed", r"([A-Z][a-z]+(?:\s+[A-Z][a-z]+)?)\s*/\s*([A-Z][a-z]+(?:\s+[A-Z][a-z]+)?)", 0.9)]))

# URL / domain recognizer
analyzer.registry.add_recognizer(PatternRecognizer(
    supported_entity="URL",
    patterns=[Pattern("url", r"(?:https?://)?(?:www\.)?[a-zA-Z0-9\-]+\.[a-z]{2,}(?:\.[a-z]{2,})?(?:/[^\s]*)?", 0.85)]))

# ── Target entities (full assignment scope) ─────────────────────────────────
TARGET_ENTITIES = [
    "PERSON", "ORGANIZATION", "LOCATION", "EMAIL_ADDRESS",
    "PHONE_NUMBER", "IN_PAN", "IN_AADHAAR", "DATE_OF_BIRTH",
    "IP_ADDRESS", "IN_CIN", "IN_SEBI_REG", "IN_PIN_CODE",
    "US_SSN", "CREDIT_CARD", "URL"
]

# ── Allowlist: terms that MUST NOT be redacted ──────────────────────────────
# These are public regulatory bodies, stock exchanges, statutory terms,
# and newspaper names that appear in boilerplate legal text.
ALLOWLIST = {
    # Countries / states (regulatory context, not personal addresses)
    "INDIA", "MAHARASHTRA", "MUMBAI", "PUNE", "DELHI", "NEW DELHI",
    # Regulators & exchanges
    "SEBI", "BSE", "NSE", "ROC", "RBI", "ICICI", "HDFC",
    "BSE LIMITED", "NATIONAL STOCK EXCHANGE OF INDIA LIMITED",
    "SECURITIES AND EXCHANGE BOARD OF INDIA",
    "RESERVE BANK OF INDIA", "REGISTRAR OF COMPANIES",
    "CENTRAL PROCESSING CENTRE",
    # Statutory boilerplate
    "COMPANIES ACT", "GOVERNMENT OF INDIA", "CENTRAL GOVERNMENT",
    "QUALIFIED INSTITUTIONAL BUYERS", "QUALIFIED INSTITUTIONAL BUYER",
    "QIB", "QIBS", "NON-INSTITUTIONAL INVESTORS", "NII",
    "RETAIL INDIVIDUAL INVESTORS",
    # Newspaper names (publication clauses)
    "FINANCIAL EXPRESS", "JANSATTA", "LOKSATTA",
    # Offer-specific date/timeline terms (must never be redacted)
    "BID/OFFER OPENING DATE", "BID/OFFER CLOSING DATE",
    "BID/OFFER PERIOD", "BID/OFFER CLOSING DAY",
    "ANCHOR INVESTOR BIDDING DATE", "WORKING DAY", "PRICING DATE",
    # Corporate officer titles & designations (not personal names)
    "BOARD OF DIRECTORS", "COMPANY SECRETARY",
    "CHIEF EXECUTIVE OFFICER", "CHIEF FINANCIAL OFFICER",
    "MANAGING DIRECTOR", "WHOLE TIME DIRECTOR",
    "INDEPENDENT DIRECTOR", "NOMINEE DIRECTOR",
    "CHAIRMAN", "CHAIRPERSON",
    # Offer-specific date/timeline terms (must never be redacted)
    "BID/OFFER OPENING DATE", "BID/OFFER CLOSING DATE",
    "BID/OFFER PERIOD", "BID/OFFER CLOSING DAY",
    "ANCHOR INVESTOR BIDDING DATE", "WORKING DAY", "PRICING DATE",
    # Corporate officer titles & designations (not personal names)
    "BOARD OF DIRECTORS", "COMPANY SECRETARY",
    "CHIEF EXECUTIVE OFFICER", "CHIEF FINANCIAL OFFICER",
    "MANAGING DIRECTOR", "WHOLE TIME DIRECTOR",
    "INDEPENDENT DIRECTOR", "NOMINEE DIRECTOR",
    "CHAIRMAN", "CHAIRPERSON",
}

print(f"Presidio engine ready.  {len(TARGET_ENTITIES)} entity types active.  Allowlist: {len(ALLOWLIST)} terms.")

## 4. Faker Synthetic-Data Registry

Every real PII string maps to exactly **one** fake replacement via `PII_REGISTRY`,
so the same name always becomes the same fake name throughout the document.

In [ ]:
fake = Faker("en_US")
PII_REGISTRY = {}


def generate_matched_email(original_email: str) -> str:
    """Build a fake email whose local-part mirrors name substitutions already in the registry."""
    if "@" not in original_email:
        return fake.user_name() + "@" + fake.free_email_domain()

    local_part, _ = original_email.split("@", 1)
    local_lower = local_part.lower()
    matched = False

    for orig_val, fake_val in PII_REGISTRY.items():
        orig_parts = orig_val.lower().split()
        fake_parts = fake_val.lower().split()
        for i, o_part in enumerate(orig_parts):
            if len(o_part) > 2 and o_part in local_lower:
                f_part = fake_parts[i] if i < len(fake_parts) else fake_parts[0]
                local_lower = local_lower.replace(o_part, f_part)
                matched = True

    if not matched:
        return fake.user_name() + "@" + fake.free_email_domain()
    return local_lower + "@" + fake.free_email_domain()


def get_consistent_fake(original_value: str, entity_type: str) -> str:
    """Return a deterministic fake value for *original_value*, creating one if first seen."""
    cleaned = original_value.strip()
    if cleaned in PII_REGISTRY:
        return PII_REGISTRY[cleaned]

    generators = {
        "PERSON":       lambda: fake.name(),
        "ORGANIZATION": lambda: fake.company(),
        "LOCATION":     lambda: fake.address().replace("\n", ", "),
        "EMAIL_ADDRESS":lambda: generate_matched_email(original_value),
        "PHONE_NUMBER": lambda: fake.phone_number(),
        "IN_PAN":       lambda: f"[PAN_REDACTED_{fake.bothify('????####?')}]",
        "IN_AADHAAR":   lambda: f"[AADHAAR_REDACTED_{fake.numerify('#### #### ####')}]",
        "DATE_OF_BIRTH":lambda: fake.date_of_birth(minimum_age=21, maximum_age=70).strftime("%d/%m/%Y"),
        "IP_ADDRESS":   lambda: fake.ipv4(),
        "IN_CIN":       lambda: f"[CIN_REDACTED_{fake.bothify('?#####??####???######')}]",
        "IN_SEBI_REG": lambda: f"[SEBI_REDACTED_{fake.bothify('IN???#########')}]",
        "IN_PIN_CODE":  lambda: fake.postcode(),
        "US_SSN":       lambda: fake.ssn(),
        "CREDIT_CARD": lambda: fake.credit_card_number(),
        "URL":          lambda: "www.example-company.com",
    }

    new_fake = generators.get(entity_type, lambda: fake.word())()
    PII_REGISTRY[cleaned] = new_fake
    return new_fake

## 5. Core Redaction Engine

Three functions:
1. `redact_text_block` — analyse a single text string, replace PII.
2. `process_image_ocr` — OCR an embedded image, then redact the extracted text.
3. `process_docx` — orchestrate paragraphs → tables → images for the whole `.docx`.

In [ ]:
def redact_text_block(text: str, analyzer_engine):
    """Analyse *text* and return (redacted_text, [(original, entity_type), ...])."""
    if not text or not text.strip():
        return text, []

    # Sentence-case all-caps blocks so SpaCy doesn't hallucinate every word as PERSON
    text_to_analyze = text.capitalize() if text.isupper() and len(text) > 20 else text

    results = analyzer_engine.analyze(
        text=text_to_analyze, entities=TARGET_ENTITIES, language="en"
    )

    # ── Post-filter: allowlist + SEBI false-positive guard ──────────────────
    valid = []
    for r in results:
        span_text = text[r.start : r.end]

        # Skip anything in the allowlist
        if span_text.upper().strip() in ALLOWLIST:
            continue

        # SEBI IDs must be uppercase in the original
        if r.entity_type == "IN_SEBI_REG" and not span_text.isupper():
            continue

        # PIN codes: only keep if near an address-like keyword (reduces FP on random 6-digit numbers)
        if r.entity_type == "IN_PIN_CODE":
            context = text[max(0, r.start - 80) : r.end + 20].upper()
            if not any(kw in context for kw in ["PIN", "ADDRESS", "OFFICE", "ROAD", "STREET", "NAGAR", "COLONY", "WARD"]):
                continue

        valid.append(r)

    # ── Resolve overlapping spans: keep higher-confidence match ─────────────
    valid.sort(key=lambda x: x.start)
    filtered = []
    for cur in valid:
        if filtered and cur.start < filtered[-1].end:
            if cur.score > filtered[-1].score:
                filtered[-1] = cur
        else:
            filtered.append(cur)

    # ── Replace spans right-to-left to preserve character offsets ───────────
    filtered.sort(key=lambda x: x.start, reverse=True)
    redacted = text
    detections = []
    for r in filtered:
        original = text[r.start : r.end]
        detections.append((original, r.entity_type))
        replacement = get_consistent_fake(original, r.entity_type)
        redacted = redacted[: r.start] + replacement + redacted[r.end :]

    return redacted, detections


def process_image_ocr(image_bytes: bytes):
    """OCR an image and redact the extracted text."""
    try:
        image = Image.open(io.BytesIO(image_bytes))
        raw_text = pytesseract.image_to_string(image)
        # Unglue OCR artefacts (e.g. 'nameVISHAL' → 'name VISHAL')
        raw_text = re.sub(r"([a-z])([A-Z])", r"\1 \2", raw_text)
        return redact_text_block(raw_text, analyzer)
    except Exception:
        return "", []


def process_docx(input_path: str, output_path: str):
    """Read a .docx, redact all PII in paragraphs / tables / images, and save."""
    doc = docx.Document(input_path)
    all_detections = []

    # ── Paragraphs ─────────────────────────────────────────────────────────
    for p in doc.paragraphs:
        redacted, dets = redact_text_block(p.text, analyzer)
        p.text = redacted
        all_detections.extend(dets)

    # ── Tables ─────────────────────────────────────────────────────────────
    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                for p in cell.paragraphs:
                    redacted, dets = redact_text_block(p.text, analyzer)
                    p.text = redacted
                    all_detections.extend(dets)

    # ── Embedded images (OCR + visual redaction + QR code blanking) ───────────────────────────
    for part in doc.part.related_parts.values():
        if part.content_type.startswith("image/"):
            image_bytes = part.blob
            redacted_ocr, dets = process_image_ocr(image_bytes)
            all_detections.extend(dets)
            try:
                # Replace ALL images (ID cards, QR codes, logos) with blank canvas.
                # QR codes are especially dangerous: a scannable QR
                # bypasses all text-level anonymisation.
                img = Image.open(io.BytesIO(image_bytes))
                canvas = Image.new("RGB", img.size, (255, 255, 255))
                draw = ImageDraw.Draw(canvas)
                if redacted_ocr and redacted_ocr.strip():
                    draw.text((10, 10), redacted_ocr[:1000], fill=(0, 0, 0))
                else:
                    # Image with no OCR text (QR code, logo, decorative)
                    draw.text((10, 10), "[IMAGE REDACTED]", fill=(128, 128, 128))
                buf = io.BytesIO()
                canvas.save(buf, format=img.format or "PNG")
                part._blob = buf.getvalue()
            except Exception:
                pass

    doc.save(output_path)
    print(f"Anonymised document saved → {output_path}")
    return all_detections

## 6. Evaluation Function & Human-Verified Ground Truth

Ground truth was **manually read** from the first 5 pages of `Red Herring Prospectus.docx`.
Keys use standard Presidio entity types so `score()` can directly consume the
`(text, label)` tuples produced by the pipeline.

In [ ]:
# ── Human-checked ground truth (first 5 pages of the DRHP) ─────────────────
GROUND_TRUTH = {
    "PERSON": [
        "Kushal Subbayya Hegde",
        "Pushpa Kushal Hegde",
        "Rajesh Kushal Hegde",
        "Rohit Kushal Hegde",
        # OCR-extracted names from embedded PAN / Aadhaar images
        "VISHAL SINGH",
        "SUGRIV SINGH",
        "MERAJ KHAN",
        "Sudhdan Khan",
    ],
    "IN_PAN": [
        "NBWPS1951N",
    ],
    "IN_AADHAAR": [
        "2943 6593 3461",
    ],
    "DATE_OF_BIRTH": [
        "06/05/2000",
        "12/12/1988",
    ],
}


def score(detections: list, ground_truth: dict = GROUND_TRUTH) -> dict:
    """
    Compute per-entity-type and overall Precision, Recall, F1.

    *detections* is the raw list of (text, entity_type) tuples from process_docx.
    Matching is case-insensitive and stripped of whitespace.
    """
    # Bucket predictions by entity type
    pred_by_type = {k: [] for k in ground_truth}
    for text, label in detections:
        if label in pred_by_type:
            pred_by_type[label].append(text)

    results = {}
    total_tp = total_fp = total_fn = 0

    for etype, truth_list in ground_truth.items():
        truth_set = {t.strip().lower() for t in truth_list}
        pred_set  = {p.strip().lower() for p in pred_by_type[etype]}

        tp = len(truth_set & pred_set)
        fp = len(pred_set - truth_set)
        fn = len(truth_set - pred_set)

        prec = tp / (tp + fp) if (tp + fp) else 1.0
        rec  = tp / (tp + fn) if (tp + fn) else 1.0
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0

        results[etype] = {
            "TP": tp, "FP": fp, "FN": fn,
            "precision": round(prec, 3),
            "recall":    round(rec, 3),
            "f1_score":  round(f1, 3),
        }
        total_tp += tp; total_fp += fp; total_fn += fn

    p = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 1.0
    r = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 1.0
    f = 2 * p * r / (p + r) if (p + r) else 0.0

    results["OVERALL"] = {
        "TP": total_tp, "FP": total_fp, "FN": total_fn,
        "precision": round(p, 3),
        "recall":    round(r, 3),
        "f1_score":  round(f, 3),
    }
    return results

## 7. Execute Pipeline & Print Evaluation Report

In [ ]:
INPUT_FILE  = "Red Herring Prospectus.docx"
OUTPUT_FILE = "Anonymized_Red_Herring_Prospectus.docx"

# ── Run redaction ──────────────────────────────────────────────────────────
all_detections = process_docx(INPUT_FILE, OUTPUT_FILE)

# ── Run stats ──────────────────────────────────────────────────────────────
counts = Counter(label for _, label in all_detections)
print(f"\nTotal entities redacted: {len(all_detections)}")
for label, n in counts.most_common():
    print(f"  {label:20s}  {n}")

# ── Evaluation against human-checked ground truth ──────────────────────────
report = score(all_detections)

print("\n" + "=" * 60)
print("  EVALUATION REPORT  (Ground Truth = first 5 pages, human-checked)")
print("=" * 60)
df = pd.DataFrame(report).T
df.index.name = "Entity Type"
print(df.to_string())
print("\nRaw JSON:")
print(json.dumps(report, indent=2))